In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\CCR_DATA_2023_25\Raw_data_1Day_2024_site_122_Mandir_Marg_Delhi_DPCC_1Day.csv")

In [3]:
df

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),...,MP-Xylene (µg/m³),AT (°C),RH (%),WS (m/s),WD (deg),RF (mm),TOT-RF (mm),SR (W/mt2),BP (mmHg),VWS (m/s)
0,2024-01-01,177.67,284.38,29.41,46.54,75.95,79.16,2.45,0.59,4.35,...,NaN,9.72,81.48,0.80,245.24,NaN,0.0,29.33,975.85,0.00
1,2024-01-02,166.63,273.08,32.23,47.33,79.56,77.48,2.79,0.63,4.51,...,NaN,8.90,78.04,0.70,239.51,NaN,0.0,36.92,976.01,0.00
2,2024-01-03,179.15,272.59,33.75,47.96,81.71,80.88,2.66,0.61,4.04,...,NaN,7.68,88.79,0.60,232.63,NaN,0.0,26.05,975.89,0.00
3,2024-01-04,219.97,337.91,43.13,46.84,89.97,83.18,2.75,1.12,1.51,...,NaN,8.12,89.82,0.97,249.42,NaN,0.0,13.38,975.93,0.00
4,2024-01-05,171.10,301.54,41.56,45.42,86.98,85.41,2.22,0.49,1.09,...,NaN,9.52,92.21,0.59,247.42,NaN,0.0,10.70,975.86,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,2024-12-27,153.08,170.64,26.08,102.80,75.88,60.60,5.94,1.44,32.39,...,NaN,15.91,93.28,0.57,200.55,NaN,0.0,6.53,986.00,-0.02
362,2024-12-28,104.35,128.07,32.74,101.59,80.67,53.07,4.03,1.41,25.89,...,NaN,16.89,94.92,0.46,259.00,NaN,0.0,10.86,986.00,-0.02
363,2024-12-29,97.92,120.78,14.61,70.49,49.38,50.41,4.35,1.32,32.34,...,NaN,15.97,90.43,1.31,255.02,NaN,0.0,25.52,986.00,-0.02
364,2024-12-30,93.17,113.71,9.65,56.24,37.76,45.80,4.86,1.92,33.93,...,NaN,13.00,88.87,1.04,255.83,NaN,0.0,19.14,986.00,-0.02


In [4]:
# ---------- 2. Remove duplicate rows and columns ----------
df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[:, ~df.T.duplicated()]
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (366, 22)


In [5]:
# ---------- 3. Handle missing values (drop >70% NaN, impute median otherwise) ----------
nan_thresh = 0.7

# Drop columns with >70% missing
cols_to_drop = df.columns[df.isnull().mean() > nan_thresh]
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns (>{int(nan_thresh*100)}% NaN): {cols_to_drop.tolist()}")

# Drop rows with >70% missing
rows_to_drop = df.index[df.isnull().mean(axis=1) > nan_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
print(f"Dropped rows (>{int(nan_thresh*100)}% NaN):", len(rows_to_drop))

# Impute remaining missing values
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val[0])

print("Missing values after imputation:\n", df.isnull().sum())

Dropped columns (>70% NaN): ['Xylene (µg/m³)', 'O Xylene (µg/m³)']
Dropped rows (>70% NaN): 17
Missing values after imputation:
 Timestamp          0
PM2.5 (µg/m³)      0
PM10 (µg/m³)       0
NO (µg/m³)         0
NO2 (µg/m³)        0
NOx (ppb)          0
NH3 (µg/m³)        0
SO2 (µg/m³)        0
CO (mg/m³)         0
Ozone (µg/m³)      0
Benzene (µg/m³)    0
Toluene (µg/m³)    0
AT (°C)            0
RH (%)             0
WS (m/s)           0
WD (deg)           0
TOT-RF (mm)        0
SR (W/mt2)         0
BP (mmHg)          0
VWS (m/s)          0
dtype: int64


In [6]:
# ---------- 4. Handle outliers using IQR (with 70% rule) ----------

def get_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Handle columns: drop if >70% outliers, else replace with median
outlier_thresh = 0.7
cols_to_drop = []
for col in num_cols:
    outlier_mask = get_outlier_mask(df[col])
    outlier_fraction = outlier_mask.mean()
    if outlier_fraction > outlier_thresh:
        cols_to_drop.append(col)
    else:
        median_val = df[col].median()
        df.loc[outlier_mask, col] = median_val
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped numeric columns (>{int(outlier_thresh*100)}% outliers): {cols_to_drop}")
    
# Handle rows: drop if >70% numeric columns are outliers in a given row
outlier_matrix = df[num_cols].apply(get_outlier_mask)
row_outlier_fraction = outlier_matrix.mean(axis=1)
rows_to_drop = df.index[row_outlier_fraction > outlier_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
if len(rows_to_drop) > 0:
    print(f"Dropped rows (>{int(outlier_thresh*100)}% outliers): {len(rows_to_drop)}")


In [7]:
# ---------- 6. Final check ----------
print("Final shape:", df.shape)
print(df.head())

Final shape: (349, 20)
    Timestamp  PM2.5 (µg/m³)  PM10 (µg/m³)  NO (µg/m³)  NO2 (µg/m³)  \
0  2024-01-01         177.67        284.38       29.41        46.54   
1  2024-01-02         166.63        273.08       32.23        47.33   
2  2024-01-03         179.15        272.59       33.75        47.96   
3  2024-01-04         219.97        337.91       43.13        46.84   
4  2024-01-05         171.10        301.54       41.56        45.42   

   NOx (ppb)  NH3 (µg/m³)  SO2 (µg/m³)  CO (mg/m³)  Ozone (µg/m³)  \
0      75.95        27.37         2.45        0.59           4.35   
1      79.56        27.37         2.79        0.63           4.51   
2      81.71        27.37         2.66        0.61           4.04   
3      89.97        27.37         2.75        1.12           1.51   
4      86.98        27.37         2.22        0.49           1.09   

   Benzene (µg/m³)  Toluene (µg/m³)  AT (°C)  RH (%)  WS (m/s)  WD (deg)  \
0            2.055             4.68     9.72   81.48      0

In [8]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerics = df.select_dtypes(include=[np.number]).columns
df[numerics] = scaler.fit_transform(df[numerics])

In [9]:
df

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),Benzene (µg/m³),Toluene (µg/m³),AT (°C),RH (%),WS (m/s),WD (deg),TOT-RF (mm),SR (W/mt2),BP (mmHg),VWS (m/s)
0,2024-01-01,1.315788,1.681319,-0.515238,-0.904628,0.152060,-0.271125,-2.338598,-1.148885,-1.346513,0.0,0.0,-1.933161,0.484238,-0.113503,0.637518,0.0,-0.891074,-1.675480,-3.469447e-18
1,2024-01-02,1.154924,1.543799,-0.385778,-0.876801,0.299213,-0.271125,-2.173958,-1.057289,-1.333443,0.0,0.0,-2.038204,0.304248,-0.380614,0.495741,0.0,-0.677331,-1.640076,-3.469447e-18
2,2024-01-03,1.337353,1.537836,-0.315998,-0.854611,0.386853,-0.271125,-2.236909,-1.103087,-1.371835,0.0,0.0,-2.194489,0.866717,-0.647725,0.325511,0.0,-0.983442,-1.666629,-3.469447e-18
3,2024-01-04,1.932142,2.332773,0.114617,-0.894061,0.723552,-0.271125,-2.193328,0.064760,-1.578502,0.0,0.0,-2.138124,0.920610,0.340586,0.740942,0.0,-1.340243,-1.657778,-3.469447e-18
4,2024-01-05,1.220056,1.890154,0.042542,-0.944079,0.601672,-0.271125,-2.449971,-1.377875,-1.612811,0.0,0.0,-1.958781,1.045661,-0.674436,0.691457,0.0,-1.415714,-1.673267,-3.469447e-18
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
344,2024-12-27,0.957487,0.297116,-0.668111,1.077059,0.149207,1.946471,-0.648623,0.797527,0.943978,0.0,0.0,-1.140210,1.101646,-0.727859,-0.468238,0.0,-1.533146,0.570449,-3.469447e-18
345,2024-12-28,0.247441,-0.220957,-0.362365,1.034438,0.344460,1.443958,-1.573509,0.728830,0.413016,0.0,0.0,-1.014670,1.187455,-1.021681,0.977978,0.0,-1.411209,0.570449,-3.469447e-18
346,2024-12-29,0.153749,-0.309675,-1.194673,-0.061020,-0.931004,1.266444,-1.418554,0.522740,0.939894,0.0,0.0,-1.132524,0.952526,1.248764,0.879502,0.0,-0.998367,0.570449,-3.469447e-18
347,2024-12-30,0.084537,-0.395716,-1.422376,-0.562958,-1.404666,0.958796,-1.171595,1.896678,1.069775,0.0,0.0,-1.512987,0.870903,0.527564,0.899544,0.0,-1.178035,0.570449,-3.469447e-18


In [10]:
df.to_excel('mandirmrg2024.xlsx', index=False)